# ScholarRAG: query rewriting for broad questions

**Problem (M6 testing):** a conversational overview question such as "I am interested to study about the unsupervised medical anomaly detection, what are their current research?" is refused, because "unsupervised anomaly detection" dominates the wording and the five retrieved passages are industrial. The corpus does contain medical papers.

**Idea:** before retrieval, an LLM (`gemini-3.5-flash-lite`) rewrites the question into a search query that leads with the topic and adds synonyms (prompt `REWRITE_SYSTEM` in `src/rag.py`, version r2). Retrieval then pools candidates for the question **and** for the search query.

**Test:** five broad questions on different topics. For each, count how many of the five final passages are on topic: a keyword for the topic appears in the paper title or the first 300 characters of the passage. This is a rough proxy, read together with the titles.

| Variant | Candidate pool | Reranker compares passages with |
|---|---|---|
| No rewriting | question (the M3 setup) | question |
| Union, rerank question | question ∪ search query | question |
| Union, rerank search query | question ∪ search query | search query |

Rewrites are cached in `eval/m6_rewrites.jsonl`, because the same prompt can return a different query on another run.

In [1]:
import os, re, sys, json, time
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

ROOT = Path.home() / "Projects" / "scholar-rag"
sys.path.insert(0, str(ROOT / "src"))
import rag
load_dotenv(ROOT / ".env")
from google import genai

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
ch = rag.load_chunks()
R = rag.Retriever(ch)

BROAD = [
    ("medical", r"medical|\bMRI\b|brain|patholog|X-ray|retina|ultrasound|clinical|\bCT\b",
     "I am interested to study about the unsupervised medical anomaly detection, what are their current research?"),
    ("video", r"video|surveillance", "I'm curious how people find anomalies in surveillance videos, what approaches exist?"),
    ("logical", r"logical|LOCO", "what are the main ways to detect logical anomalies, like missing or misplaced parts?"),
    ("3D", r"\b3D\b|point cloud", "Can you tell me what is being done for 3D point cloud anomaly detection lately?"),
    ("dataset", r"dataset|benchmark", "what datasets do researchers use to benchmark industrial anomaly detection?"),
]

W1026 19:51:40.976000 27099 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [2]:
cache_path = rag.BASE / "eval" / "m6_rewrites.jsonl"
cache = {}
if cache_path.exists():
    for line in open(cache_path):
        r = json.loads(line)
        cache[(r["question"], r["version"])] = r
with open(cache_path, "a") as f:
    for _, _, q in BROAD:
        if (q, rag.REWRITE_VERSION) not in cache:
            query, secs = rag.rewrite_query(client, q)
            rec = {"question": q, "version": rag.REWRITE_VERSION, "query": query, "seconds": secs}
            cache[(q, rag.REWRITE_VERSION)] = rec
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
            time.sleep(4.5)
pd.DataFrame([{"topic": t, "search query": cache[(q, rag.REWRITE_VERSION)]["query"],
               "rewrite seconds": cache[(q, rag.REWRITE_VERSION)]["seconds"]} for t, _, q in BROAD])

,topic,search query,rewrite seconds
0,medical,unsupervised medical anomaly detection MRI CT ...,9.98
1,video,video surveillance anomaly detection video ano...,10.74
2,logical,logical anomalies missing misplaced parts stru...,26.31
3,3D,3D point cloud anomaly detection point clouds ...,10.38
4,dataset,industrial anomaly detection datasets benchmar...,24.53


In [3]:
rows, titles = [], {}
for topic, pat, q in BROAD:
    sq = cache[(q, rag.REWRITE_VERSION)]["query"]
    on = lambda idxs: sum(bool(re.search(pat, ch.title.iat[i] + " " + ch.text.iat[i][:300], re.I)) for i in idxs)
    variants = {"no rewriting": R.retrieve(q)[0],
                "union, rerank question": R.retrieve(q, search_query=sq, rerank_with="question")[0],
                "union, rerank search query": R.retrieve(q, search_query=sq, rerank_with="search")[0]}
    rows.append({"topic": topic, **{k: on(v) for k, v in variants.items()}})
    titles[topic] = {k: [ch.title.iat[i][:60] for i in v] for k, v in variants.items()}
res = pd.DataFrame(rows).set_index("topic")
res.loc["total (of 25)"] = res.sum()
res

,no rewriting,"union, rerank question","union, rerank search query"
topic,,,
medical,1,1,4
video,5,5,5
logical,4,4,4
3D,5,5,5
dataset,3,3,2
total (of 25),18,18,20


In [4]:
# judul paper di 5 passage, untuk membaca angka di atas dengan benar
for topic, tv in titles.items():
    print(f"\n[{topic}]")
    for k, ts in tv.items():
        print(f"  {k:28s}", " | ".join(ts))


[medical]
  no rewriting                 Prior Normality Prompt Transformer for Multi-class Industria | Visual Anomaly Detection for Images: A Survey | SoftPatch+: Fully Unsupervised Anomaly Classification and Se | PKU-GoodsAD: A Supermarket Goods Dataset for Unsupervised An | Unsupervised Anomaly Detection with an Enhanced Teacher for 
  union, rerank question       Prior Normality Prompt Transformer for Multi-class Industria | Visual Anomaly Detection for Images: A Survey | SoftPatch+: Fully Unsupervised Anomaly Classification and Se | PKU-GoodsAD: A Supermarket Goods Dataset for Unsupervised An | Unsupervised Anomaly Detection with an Enhanced Teacher for 
  union, rerank search query   A Study of Representational Properties of Unsupervised Anoma | A Study of Representational Properties of Unsupervised Anoma | Anomaly Detection by Clustering DINO Embeddings using a Diri | PKU-GoodsAD: A Supermarket Goods Dataset for Unsupervised An | Visual Anomaly Detection for Images: A Survey

[

## Reading the results

| Topic | No rewriting | Union, rerank question | Union, rerank search query |
|---|---|---|---|
| medical | 1 | 1 | **4** |
| video | 5 | 5 | 5 |
| logical | 4 | 4 | 4 |
| 3D | 5 | 5 | 5 |
| dataset | 3 | 3 | **2** |
| **total (of 25)** | 18 | 18 | **20** |

- **Most broad questions already work without rewriting** (video, logical, 3D). The failure is specific to questions whose topic word ("medical") is drowned by the words every paper shares.
- **Pooling alone does nothing:** when the reranker still compares passages with the original question, it picks the same industrial passages. The gain only appears when the reranker uses the rewritten query: medical goes from 1 to 4 on-topic passages (two brain-MRI chunks lead), while dataset drops from 3 to 2.
- **Cost:** one extra Gemini call took 10–26 s here (Gemini Flash-Lite was slow that day; the answer call usually takes 1–3 s), and the same prompt can return a different query on another run.

**Decision:** query rewriting stays **off by default** in the app and is offered as an opt-in "broaden search" switch (union pool, rerank with the search query). Set A and B1 scores are therefore unchanged. When the system refuses, the UI suggests naming the topic more specifically, which fixed the medical question in M6 testing ("…in medical images such as brain MRI") at no extra cost.